# 5장 — 창발일까 착시일까 🪜

"규모를 넘자 능력이 **갑자기** 나타났다"는 창발(emergence) 주장은 매끈한 스케일링
법칙(2~4장)과 충돌합니다. 착시론의 핵심 논증을 **신의 시점**으로 재현해 봅니다:

- 밑바탕: 자리당 정답률 p(N)가 **매끈한 멱법칙**으로 개선 (급변 없음, 우리가 정한 규칙)
- 과제: 8자리 덧셈. 채점은 **8자리 전부 맞아야 정답** (전부-정답 지표)
- 질문: 같은 모델을 세 개의 안경(지표)으로 보면 각각 어떻게 보이나?

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 여기서 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch5')

## 1. 밑바탕: 매끈한 멱법칙 (급변은 없다)

자리당 오차 `err(N) = 0.85 · (N/10⁷)^(−0.3)`. 3장에서 잰 것과 같은 꼴의 직선입니다.

In [ ]:
import numpy as np

K = 8  # 답의 자릿수

def err(n):
    return 0.85 * (n / 1e7) ** -0.3

def p(n):
    return 1 - err(n)

NS = [1e7, 1e8, 1e9, 1e10, 1e11, 1e12]
print('      N | 자리당 정답률 p | 전부 정답 p^8')
print('-' * 46)
for n in NS:
    print(f'{n:>8.0e} | {p(n)*100:12.1f}% | {p(n)**K*100:10.1f}%')

# 웹 랩·본문 수치 못박기
assert abs(p(1e7) - 0.15) < 1e-12
assert round(p(1e9) * 100, 1) == 78.6 and round(p(1e9) ** K * 100, 1) == 14.6
assert round(p(1e10) * 100, 1) == 89.3 and round(p(1e10) ** K * 100, 1) == 40.4
assert round(p(1e12) * 100, 1) == 97.3 and round(p(1e12) ** K * 100, 1) == 80.4
assert round(0.9 ** 8, 2) == 0.43, '자리당 90% 실력 = 전부-정답 43점'
print()
print('✔ 밑바탕 p는 15% → 97%로 내내 매끈. 그런데 p^8 은 10⁹→10¹⁰에서 15%→40% 점프')

## 2. 세 개의 안경: 같은 모델, 다른 곡선

① 전부-정답(계단, 창발처럼) ② 자리당 정답률(완만) ③ 초과 오차 로그-로그(직선).

In [ ]:
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False
except Exception:
    pass

n_grid = np.logspace(7, 12, 200)
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))

axes[0].semilogx(n_grid, p(n_grid) ** K * 100, color='#e11d48', lw=2)
axes[0].set_title('① 전부 정답 (p⁸): "창발"처럼 보임')
axes[0].set_ylabel('%')

axes[1].semilogx(n_grid, p(n_grid) * 100, color='#10b981', lw=2)
axes[1].set_title('② 자리당 정답률 (p): 완만')

axes[2].loglog(n_grid, err(n_grid), color='#6366f1', lw=2)
axes[2].set_title('③ 초과 오차: 로그-로그 직선 (기울기 −0.3)')

for ax in axes:
    ax.set_xlabel('모델 크기 N')
plt.tight_layout()
plt.show()

# ③이 정말 직선(멱법칙)인지 — 2장의 기술로 기울기 실측
slope, _ = np.polyfit(np.log10(n_grid), np.log10(err(n_grid)), 1)
assert abs(slope - (-0.3)) < 1e-9
print(f'✔ 밑바탕 기울기 실측 = {slope:.3f}. 계단을 만든 건 모델이 아니라 지표(p^8)였다')

## 3. 계단의 제조법: 거듭제곱의 비선형 증폭

p가 1.6배 좋아질 때 p⁸은 몇 배가 되는지, 그리고 자릿수(지수)가 커지면 계단이 어떻게 변하는지.

In [ ]:
p1, p2 = p(1e8), p(1e10)
amp = (p2 ** K) / (p1 ** K)
print(f'p: {p1:.3f} → {p2:.3f} ({p2/p1:.1f}배)  |  p^8: {p1**K*100:.1f}% → {p2**K*100:.1f}% ({amp:.0f}배!)')
assert p2 / p1 < 1.6 and amp > 30, '1.6배 개선이 33배 점프로 증폭'

# 자릿수가 길수록 계단이 늦고 가파르다 — k=4 vs 8 vs 16 의 "50% 도달 규모"
for k in [4, 8, 16]:
    # p^k = 0.5 → p = 0.5^(1/k) → err = 1 − p → N 역산
    p_need = 0.5 ** (1 / k)
    n_need = 1e7 * ((1 - p_need) / 0.85) ** (1 / -0.3)
    print(f'k={k:>2}자리: 전부-정답 50% 도달에 N ≈ {n_need:.1e}')
print()
print('✔ 과제가 길수록(k↑) "창발 시점"이 뒤로 밀린다. 지표만으로 설명되는 예측')

## 정리: 오늘 가져갈 세 문장

1. **지표가 계단을 만들 수 있다**: 매끈한 p도 전부-정답(p⁸)으로 채점하면 창발처럼 보인다.
2. 착시론은 강력한 경고이지 전면 부정이 아니다. **열린 문제**로 남아 있다.
3. 급변 주장 검증 3단계: ① 지표가 비선형인지 ② 부분 점수로 다시 ③ 로그-로그 오차 확인.
   셋 다 꺾여야 진짜.

성장의 법칙 1막 끝. 다음 **6장(시험의 오차범위)**: 워밍업의 소음을 드디어 공식으로
계산합니다(3부 8장 신뢰구간 복습).